# Предзащитное обучение моделей SpectrumAI на 2000 NIST-спектрах

Этап 19 фазы 2 (`DEVELOPMENT_PLAN.md`). Запуск в Kaggle Notebook с T4 GPU.

**Что нужно подготовить заранее:**
1. Загрузить `ml/data/processed/predefense_labeled.parquet` в Kaggle Dataset с именем `spectra-id-predefense`.
2. Прикрепить dataset к этому ноутбуку (правая панель → Add Data).
3. Включить GPU T4 в настройках ноутбука.

Ожидаемое время: 1D-CNN ~2–3 ч, contrastive ~4–6 ч. На обрыв сессии есть промежуточные чекпойнты каждые 5 эпох.

## 1. Зависимости

In [ ]:
!pip install -q rdkit transformers==4.44.* structlog pyyaml

## 2. Клонирование SpectrumAI

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/SpectrumAI')
REPO_URL = 'https://github.com/Wadichka/SpectrumAI.git'
REPO_BRANCH = 'feat/predefense-real-data'
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)],
        check=True,
    )
for path in (str(REPO_DIR / 'ml'), str(REPO_DIR / 'backend')):
    if path not in sys.path:
        sys.path.insert(0, path)
print('SpectrumAI at', REPO_DIR, 'branch', REPO_BRANCH)


## 3. Загрузка датасета и сплит по InChI Key

In [ ]:
import pandas as pd

from pipelines.training.split import split_by_inchi_key

PARQUET = Path('/kaggle/input/spectra-id-predefense/predefense_labeled.parquet')
df = pd.read_parquet(PARQUET)
print('total:', len(df), 'unique molecules:', df['inchi_key'].nunique())

train_idx, val_idx, test_idx = split_by_inchi_key(
    df['inchi_key'].tolist(), ratios=(0.70, 0.15, 0.15), seed=42
)
print(f'train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')

## 4. Обучение 1D-CNN

Конфиг: `ml/configs/cnn1d_predefense.yaml`. CLI-скрипт `ml/scripts/train_cnn1d.py` принимает путь к конфигу и устройство.

Если сессия оборвалась — перезапусти ячейку: trainer резюмируется с последнего чекпойнта `/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/last.pt`.

In [ ]:
!cd /kaggle/working/SpectrumAI && python ml/scripts/train_cnn1d.py \
    --config ml/configs/cnn1d_predefense.yaml \
    --device cuda


## 5. Обучение двухбашенной (contrastive) модели

ChemBERTa загружается из HuggingFace, замораживается. Обучается только проекционная голова + spectrum tower.

In [ ]:
!cd /kaggle/working/SpectrumAI && python ml/scripts/train_contrastive.py \
    --config ml/configs/contrastive_predefense.yaml \
    --device cuda


## 6. Финальная оценка на test

Метрики (глава 11 §11.3): micro/macro F1, precision/recall, top-K retrieval recall (K=1,5,10), mAP. Результат — `/kaggle/working/output/predefense_metrics.json`.

In [ ]:
import datetime as dt

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader, Subset

from pipelines.dataset import SpectraDataset
from pipelines.labeling import GROUP_NAMES
from pipelines.metrics import compute_metrics
from pipelines.models.cnn1d import build_model as build_cnn
from pipelines.models.molecule_tower import MoleculeTower
from pipelines.models.spectrum_tower import SpectrumTower
from pipelines.retrieval_metrics import mean_reciprocal_rank, topk_accuracy

OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CNN_CKPT = Path('/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/best.pt')
CONTRASTIVE_CKPT = Path('/kaggle/working/checkpoints/contrastive-predefense-0.5.0/best.pt')

CNN_CFG = yaml.safe_load(
    Path(REPO_DIR / 'ml/configs/cnn1d_predefense.yaml').read_text(encoding='utf-8')
)
CONTRASTIVE_CFG = yaml.safe_load(
    Path(REPO_DIR / 'ml/configs/contrastive_predefense.yaml').read_text(encoding='utf-8')
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_labels = int(CNN_CFG['data']['n_labels'])
spectrum_length = int(CNN_CFG['data']['spectrum_length'])

# Test loader — без аугментаций.
test_ds = SpectraDataset(
    PARQUET,
    transform=None,
    spectrum_length=spectrum_length,
    n_labels=n_labels,
)
test_loader = DataLoader(
    Subset(test_ds, test_idx.tolist()),
    batch_size=int(CNN_CFG['training']['batch_size']),
    shuffle=False,
    num_workers=int(CNN_CFG['training'].get('num_workers', 0)),
)

# --- Classification: CNN best.pt ---
cnn = build_cnn(CNN_CFG['model']).to(device).eval()
cnn_state = torch.load(CNN_CKPT, map_location=device)
cnn.load_state_dict(cnn_state['state_dict'])
thresholds = np.asarray(cnn_state['thresholds'], dtype=np.float64)

y_true_chunks: list[np.ndarray] = []
y_prob_chunks: list[np.ndarray] = []
with torch.no_grad():
    for spectra, labels in test_loader:
        logits = cnn(spectra.to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
        y_true_chunks.append(labels.cpu().numpy())
        y_prob_chunks.append(probs)
y_true = np.concatenate(y_true_chunks, axis=0).astype(np.int64)
y_prob = np.concatenate(y_prob_chunks, axis=0).astype(np.float64)
classification_metrics = compute_metrics(y_true, y_prob, thresholds, class_names=list(GROUP_NAMES))

# --- Retrieval: контрастная модель ---
spectrum_cnn = build_cnn(CONTRASTIVE_CFG['cnn'])
spectrum_tower = SpectrumTower(
    spectrum_cnn,
    projection_dim=int(CONTRASTIVE_CFG['spectrum_tower'].get('projection_dim', 128)),
    hidden_dim=int(CONTRASTIVE_CFG['spectrum_tower'].get('projection_hidden', 256)),
    dropout=float(CONTRASTIVE_CFG['spectrum_tower'].get('projection_dropout', 0.10)),
    freeze_encoder=bool(CONTRASTIVE_CFG['spectrum_tower'].get('freeze_encoder', False)),
).to(device).eval()
contrastive_state = torch.load(CONTRASTIVE_CKPT, map_location=device)
spectrum_tower.load_state_dict(contrastive_state['spectrum_tower_state_dict'])

molecule_tower = MoleculeTower(
    contrastive_state.get(
        'molecule_model_name', CONTRASTIVE_CFG['molecule_tower'].get('model_name')
    ),
    projection_dim=int(CONTRASTIVE_CFG['molecule_tower'].get('projection_dim', 128)),
    hidden_dim=int(CONTRASTIVE_CFG['molecule_tower'].get('projection_hidden', 384)),
    dropout=float(CONTRASTIVE_CFG['molecule_tower'].get('projection_dropout', 0.10)),
    freeze_encoder=bool(CONTRASTIVE_CFG['molecule_tower'].get('freeze_encoder', True)),
    max_length=int(CONTRASTIVE_CFG['molecule_tower'].get('max_length', 128)),
).to(device).eval()
molecule_tower.projection.load_state_dict(contrastive_state['molecule_projection_state_dict'])

test_smiles = df.iloc[test_idx]['smiles'].tolist()
z_spec_parts: list[torch.Tensor] = []
z_mol_parts: list[torch.Tensor] = []
mol_batch = int(CONTRASTIVE_CFG['training'].get('batch_size', 32))

with torch.no_grad():
    for spectra, _ in test_loader:
        z_spec_parts.append(spectrum_tower(spectra.to(device)).cpu())
    for start in range(0, len(test_smiles), mol_batch):
        chunk = test_smiles[start:start + mol_batch]
        z_mol_parts.append(molecule_tower(chunk).cpu())

z_spec = torch.cat(z_spec_parts, dim=0)
z_mol = torch.cat(z_mol_parts, dim=0)
retrieval_metrics = topk_accuracy(z_spec, z_mol, ks=(1, 5, 10))
retrieval_metrics['mrr'] = mean_reciprocal_rank(z_spec, z_mol)

metrics = {
    'phase': 2,
    'data_source': 'real_nist_2000',
    'dataset_size': int(len(df)),
    'split_sizes': {
        'train': int(len(train_idx)),
        'val': int(len(val_idx)),
        'test': int(len(test_idx)),
    },
    'classification': {
        'macro_f1': classification_metrics['macro_f1'],
        'micro_f1': classification_metrics['micro_f1'],
        'weighted_f1': classification_metrics['weighted_f1'],
        'macro_precision': classification_metrics['macro_precision'],
        'macro_recall': classification_metrics['macro_recall'],
        'macro_ap': classification_metrics['macro_ap'],
        'macro_auc': classification_metrics['macro_auc'],
        'hamming_loss': classification_metrics['hamming_loss'],
        'subset_accuracy': classification_metrics['subset_accuracy'],
        'per_class': classification_metrics.get('per_class', {}),
    },
    'retrieval': retrieval_metrics,
    'thresholds': thresholds.tolist(),
    'cnn_checkpoint': str(CNN_CKPT),
    'contrastive_checkpoint': str(CONTRASTIVE_CKPT),
    'trained_at': dt.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z',
}

metrics_path = OUTPUT_DIR / 'predefense_metrics.json'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('macro_f1=%.4f  micro_f1=%.4f  top5=%.4f  mrr=%.4f' % (
    metrics['classification']['macro_f1'],
    metrics['classification']['micro_f1'],
    metrics['retrieval']['top5'],
    metrics['retrieval']['mrr'],
))
print('Saved:', metrics_path)


## 7. Что скачать

После завершения сессии скачайте в локальный `models/` репозитория:

- `/kaggle/working/checkpoints/cnn1d-predefense-0.5.0/best.pt` → `models/cnn1d-predefense-0.5.0/best.pt`
- `/kaggle/working/checkpoints/contrastive-predefense-0.5.0/best.pt` → `models/contrastive-predefense-0.5.0/best.pt`
- `/kaggle/working/output/predefense_metrics.json` → `ml/experiments/predefense/metrics.json`

Затем продолжите этап 20 (см. `docs/KAGGLE_TRAINING.md`).